### Grab all urls from sitemap

In [1]:
import requests
import xml.etree.ElementTree as ET
import gzip
from io import BytesIO

In [2]:
sitemap_index_url = "https://www.takealot.com/sitemap.xml"
response = requests.get(sitemap_index_url)
root = ET.fromstring(response.content)

In [3]:
namespace = {"ns": "http://www.sitemaps.org/schemas/sitemap/0.9"}
sitemap_urls = [
    sitemap.find("ns:loc", namespace).text
    for sitemap in root.findall("ns:sitemap", namespace)
    ]

print(sitemap_urls)




['https://www.takealot.com/sitemap_nonbook_0.xml.gz', 'https://www.takealot.com/sitemap_nonbook_1.xml.gz', 'https://www.takealot.com/sitemap_nonbook_10.xml.gz', 'https://www.takealot.com/sitemap_nonbook_11.xml.gz', 'https://www.takealot.com/sitemap_nonbook_12.xml.gz', 'https://www.takealot.com/sitemap_nonbook_13.xml.gz', 'https://www.takealot.com/sitemap_nonbook_14.xml.gz', 'https://www.takealot.com/sitemap_nonbook_15.xml.gz', 'https://www.takealot.com/sitemap_nonbook_16.xml.gz', 'https://www.takealot.com/sitemap_nonbook_17.xml.gz', 'https://www.takealot.com/sitemap_nonbook_18.xml.gz', 'https://www.takealot.com/sitemap_nonbook_19.xml.gz', 'https://www.takealot.com/sitemap_nonbook_2.xml.gz', 'https://www.takealot.com/sitemap_nonbook_20.xml.gz', 'https://www.takealot.com/sitemap_nonbook_21.xml.gz', 'https://www.takealot.com/sitemap_nonbook_22.xml.gz', 'https://www.takealot.com/sitemap_nonbook_23.xml.gz', 'https://www.takealot.com/sitemap_nonbook_24.xml.gz', 'https://www.takealot.com/site

### Filter book related sitemaps

In [4]:
book_sitemaps = [url for url in sitemap_urls if "sitemap_book_" in url.lower()]
print(len(book_sitemaps))
print(book_sitemaps[:5])

404
['https://www.takealot.com/sitemap_book_0.xml.gz', 'https://www.takealot.com/sitemap_book_1.xml.gz', 'https://www.takealot.com/sitemap_book_10.xml.gz', 'https://www.takealot.com/sitemap_book_100.xml.gz', 'https://www.takealot.com/sitemap_book_101.xml.gz']


### Process gz files

In [5]:
book_sitemap_url = book_sitemaps[0]

response = requests.get(book_sitemap_url)

print(response.status_code)
print(response.headers.get("Content-Type"))
print(response.headers.get("Content-Encoding"))
print(response.content[:100])



200
text/xml; charset=ascii
gzip
b'<?xml version="1.0" encoding="UTF-8"?>\n<urlset\n  xmlns="https://www.sitemaps.org/schemas/sitemap/0.9'


In [6]:
# with gzip.GzipFile(fileobj=BytesIO(response.content)) as gz:
#     xml_content = gz.read()

root = ET.fromstring(response.content)

namespace = {"ns": "https://www.sitemaps.org/schemas/sitemap/0.9"}



urls = [
    url.find("ns:loc", namespace).text
    for url in root.findall("ns:url", namespace)
]

print(len(urls))
print(urls[:5])


30000
['https://www.takealot.com/liefde-geduld-vriendelikheid/PLID73890439', 'https://www.takealot.com/disciple/PLID95943862', 'https://www.takealot.com/charlie-and-the-christmas-factory/PLID95408346', 'https://www.takealot.com/roald-dahl-collection-16-books-box-set/PLID70535926', 'https://www.takealot.com/shuters-premier-mental-maths-grade-6/PLID44891317']


### Grab all book urls from book_sitemap

In [9]:
all_book_urls = []

for sitemap in book_sitemaps:
    response = requests.get(sitemap)
    root = ET.fromstring(response.content)
    
    namespace = root.tag.split("}")[0].strip("{")
    ns = {"ns": namespace}
    
    urls = [
        url.find("ns:loc", ns).text
        for url in root.findall("ns:url", ns)
    ]
    
    all_book_urls.extend(urls)

# print(len(all_book_urls))

KeyboardInterrupt: 

### Grab three hundred books at random

In [8]:
import random

sample_urls = random.sample(all_book_urls, 300)
print(sample_urls[0])


https://www.takealot.com/millennial-star-108-no-13/PLID73377586


### Scrape the urls

##### Using wkipedia to find author's nationality

In [ ]:
def get_nationality(author_name):
    import re

    if not author_name:
        return None
    url = "https://en.wikipedia.org/w/api.php"

    headers = {"User-Agent": "SouthAfricanReadingTrends/1.0 aneledindili4@gmail.com"}


    params = {
        "action": "query",
        "format": "json",
        "titles": author_name,
        "prop": "revisions",
        "rvprop": "content",
        "rvslots": "main",
        "formatversion": "2"
    }

    response = requests.get(url, params=params, headers=headers)
    # print("Status:", response.status_code)
    # print("Content:", response.text[:500])

    if response.status_code != 200:
        return None
    if not response.text.strip():
        return None

    try:
        data = response.json()
    except ValueError:
        print("Not json")
        return None

    pages = data.get("query", {}).get("pages", [])
    if not pages or "revisions" not in pages[0]:
        return None
    

    page_content = pages[0]["revisions"][0]["slots"]["main"]["content"]

    match = re.search(r"\|\s*nationality\s*=\s*([^\n]+)", page_content)
    if match:
        return match.group(1).strip()
    
    return "Nationality not found"
        

In [36]:
# from bs4 import BeautifulSoup
import time
# import cloudscraper

def scrape_product(plid):
    url = f"https://api.takealot.com/rest/v-1-16-0/product-details/{plid}?platform=desktop&display_credit=true"

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json",
        "Origin": "https://www.takealot.com",
        "Referer": "https://www.takealot.com/"
    }

    r = requests.get(url, headers=headers)
    data = r.json()

    genre_option2 = []

    title = data.get("title")
    try:
        author_name = data["core"]["authors"][0]["Author"]
        
    except:
        author_name = None
    # genre_option1 = data["product_information"]["categories"]["displayable_text"]
    price = data["buybox"]["items"][0]["pretty_price"]
    genre_option2 = data["breadcrumbs"]["items"]

    languages = None
    for item in data["product_information"]["items"]:
        if item["display_name"] == "Languages":
            languages = item["displayable_text"]
            break

    publisher = None
    for item in data["product_information"]["items"]:
        if item["display_name"] == "Publisher":
            publisher = item["displayable_text"]
            break
    # scraper = cloudscraper.create_scraper()
    # r = scraper.get(url)
    # soup = BeautifulSoup(r.text, "html.parser")
    nationality = get_nationality(author_name)

    

    # print(r.status_code)
    # print(r.text[:500])

    return {
        # "plid": plid,
        "title": title,
        "price": price,
        # "genre_option2": genre_option2,
        "Languages": languages,
        "publisher": publisher,
        "Nationality": nationality
    }
data = []

for url in sample_urls[:10]:
    plid = url.split('/')[-1]
    data.append(scrape_product(plid))
    time.sleep(5) 

print(data[2])


#REDIRECT [[P. G. Wodehouse]]
{{R from modification}}
{{Short description|Publisher of travel guidebooks}}
{{About|the guidebook publisher|the film|Lonely Planet (film) {{!}}''Lonely Planet'' (film)||Lonely Planet (disambiguation)}}
{{Use Australian English|date=July 2022}}
{{Use dmy dates|date=July 2022}}
{{Infobox publisher
| name         = Lonely Planet
| image        = Lonely Planet.svg
| image_size   = 240px
| caption      = 
| parent       = Lonely Planet Global, Inc.
| status       = 
| traded_as    = 
| predecessor  = 
| founded      = 1973<ref name="guardian"/>
| founders     = {{ubl|[[Tony Wheeler]]|[[Maureen Wheeler]]}}
| successor    = 
| country      = Australia
| headquarters = [[Fort Mill, South Carolina]], U.S.
| distribution = {{unbulleted list|[[Grantham Book Service]] (UK) | [[Hachette Book Group]] (North America) | [[United Book Distributors]] (Australia) |<ref>{{Cite web|url=https://www.lonelyplanet.com/trade|title=Trade|website=Lonely Planet}}</ref>}}
| publicatio

### what genres are south african authors more writing on?
### which sa languages are writing which genres
### compare genres written for children 